# Notebook 02 — Klasifikasi: Prediksi Status Order

**Fase 2 · Minilab EduBI · Data Mining**

---

## Tujuan
Membangun model klasifikasi untuk memprediksi apakah sebuah order akan
**selesai (done)** atau **dibatalkan (cancelled)** berdasarkan fitur-fitur transaksi.

Model ini berguna bagi tim operasional untuk mendeteksi order berisiko tinggi
secara dini dan mengambil tindakan preventif.

## Alur
```
ClickHouse silver.silver_sales
    ↓
Feature Engineering (kategori → numerik, branch encoding)
    ↓
Split train/test (stratified)
    ↓
Random Forest Classifier
    ↓
Evaluasi: Accuracy, Precision, Recall, F1, ROC-AUC
    ↓
Feature Importance Analysis
    ↓
Log ke MLflow
```

## Referensi
- Breiman, L. (2001). *Random Forests*. Machine Learning, 45(1), 5–32.
- Scikit-learn: https://scikit-learn.org/stable/modules/ensemble.html#forests-of-randomized-trees

---
## 1. Setup & Koneksi

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import clickhouse_connect
import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

CH_HOST  = os.getenv('CH_HOST', 'localhost')
CH_PORT  = int(os.getenv('CH_PORT', 8123))
CH_USER  = os.getenv('CH_USER', 'default')
CH_PASS  = os.getenv('CH_PASSWORD', '')
MLFLOW_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://localhost:5000')

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('02_classification_order_status')

client = clickhouse_connect.get_client(
    host=CH_HOST, port=CH_PORT,
    username=CH_USER, password=CH_PASS
)
print('Koneksi ClickHouse berhasil.')

---
## 2. Load Data dari Silver Layer

In [ ]:
query = """
SELECT
    order_id,
    customer_id,
    category,
    quantity,
    toFloat64(unit_price)   AS unit_price,
    toFloat64(total_price)  AS total_price,
    order_year,
    order_month,
    branch,
    revenue_category,
    status
FROM silver.silver_sales
WHERE status IN ('done', 'cancelled')
"""

df = client.query_df(query)
print(f'Total records: {len(df)}')
print('\nDistribusi target (status):')
print(df['status'].value_counts())
df.head()

---
## 3. Feature Engineering

In [ ]:
df_model = df.copy()

# Target encoding
df_model['target'] = (df_model['status'] == 'done').astype(int)

# Categorical encoding
le = LabelEncoder()
for col in ['category', 'branch', 'revenue_category']:
    df_model[col + '_enc'] = le.fit_transform(df_model[col].astype(str))

# Fitur final
FEATURES = [
    'quantity', 'unit_price', 'total_price',
    'order_year', 'order_month',
    'category_enc', 'branch_enc', 'revenue_category_enc'
]
TARGET = 'target'

X = df_model[FEATURES]
y = df_model[TARGET]

print('Fitur:', FEATURES)
print('Shape X:', X.shape)
print('Distribusi target:', y.value_counts().to_dict())

---
## 4. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

---
## 5. Training & Evaluasi dengan MLflow

In [ ]:
PARAMS = {
    'n_estimators': 100,
    'max_depth':    5,
    'min_samples_leaf': 2,
    'random_state': 42
}

with mlflow.start_run(run_name='random_forest_v1'):
    model = RandomForestClassifier(**PARAMS)
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall':    recall_score(y_test, y_pred, zero_division=0),
        'f1':        f1_score(y_test, y_pred, zero_division=0),
        'roc_auc':   roc_auc_score(y_test, y_proba)
    }

    mlflow.log_params(PARAMS)
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(model, 'random_forest_model')

    for k, v in metrics.items():
        print(f'  {k:<12}: {v:.4f}')

In [ ]:
# Classification Report
print(classification_report(y_test, y_pred, target_names=['Cancelled', 'Done']))

---
## 6. Confusion Matrix & Feature Importance

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Cancelled', 'Done'],
    cmap='Blues', ax=ax1
)
ax1.set_title('Confusion Matrix')

# Feature Importance
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
importances.plot.barh(ax=ax2, color='steelblue')
ax2.set_title('Feature Importance')
ax2.set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('experiments/classification_results.png', dpi=100)
plt.show()

---
## 7. Kesimpulan

**Pertanyaan Diskusi:**
1. Fitur apa yang paling berpengaruh dalam prediksi status order?
2. Apa yang terjadi jika data sangat tidak seimbang (imbalanced)? Bagaimana mengatasinya?
3. Mengapa F1-score lebih relevan dari Accuracy untuk kasus ini?
4. Bagaimana model ini dapat diintegrasikan ke sistem operasional nyata?

**Lihat hasil eksperimen di MLflow:** http://localhost:5000